In [ ]:
"""
Acetylation PTM prediction from ProtT5 embeddings — ProteinTransformer.

Architecture (unchanged from the target spec):
    Linear(1024 -> 512) -> LayerNorm -> learned positions
    -> TransformerEncoder(3 layers, 8 heads, ff=1024, dropout=0.3, gelu)
    -> concat[center token, mean over window]   (1024 dims)
    -> Linear(1024 -> 256) -> ReLU -> Dropout(0.3) -> Linear(256 -> 1)

AdamW(lr=1e-4, weight_decay=1e-4), CosineAnnealingLR(T_max=30), batch 64.

Two switches at the top:
    BALANCE_CLASSES  pos_weight in the loss (True keeps the old behaviour)
    GROUPED_SPLIT    split by protein instead of at random — see notes below
"""

import gc
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import matthews_corrcoef, roc_auc_score, confusion_matrix

# ---------------------------------------------------------------- config

FILE_1 = "/content/drive/MyDrive/Acetylation/subash_with_all_csv_file_present_final_acetylation_graph.pt"
FILE_2 = "/content/drive/MyDrive/Acetylation/subash_with_all_csv_file_present_final_acetylation_graph_after_26169_line.pt"
CKPT = "best_model_proteintransformer.pt"

SEQ_LEN      = 25
INPUT_DIM    = 1024
HIDDEN_DIM   = 512
NHEAD        = 8
FF_DIM       = 1024
NUM_LAYERS   = 3
DROPOUT      = 0.3

BATCH_SIZE   = 64
LR           = 1e-4
WEIGHT_DECAY = 1e-4
T_MAX        = 30
NUM_EPOCHS   = 30
PATIENCE     = 5

BALANCE_CLASSES = True    # False -> plain BCEWithLogitsLoss
GROUPED_SPLIT   = True    # False -> old random stratified split
GROUP_ATTR      = "unique_id"   # attribute holding the protein identifier

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)")
print("Using device:", device)

from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------- data

data1 = torch.load(FILE_1, weights_only=False)
data2 = torch.load(FILE_2, weights_only=False)
all_data = data1[0] + data2[0]

valid_data = [item for item in all_data if item.emb.shape[0] == SEQ_LEN]
print(f"windows with {SEQ_LEN} residues: {len(valid_data)} / {len(all_data)}")

all_embs = torch.stack([item.emb for item in valid_data]).float()
all_ys = torch.tensor([item.y for item in valid_data]).float().view(-1)
print("dataset shape:", tuple(all_embs.shape))


def protein_of(item):
    """Strip the site suffix off an id like 'P12345_128' -> 'P12345'."""
    raw = str(getattr(item, GROUP_ATTR))
    return raw.rsplit("_", 1)[0] if "_" in raw else raw


groups = None
if GROUPED_SPLIT:
    if hasattr(valid_data[0], GROUP_ATTR):
        groups = np.array([protein_of(it) for it in valid_data])
        print(f"grouping by '{GROUP_ATTR}': {len(np.unique(groups))} proteins")
    else:
        print(f"WARNING: items have no '{GROUP_ATTR}' — falling back to "
              f"random split. Set GROUP_ATTR to the right field name.")

del all_data, data1, data2
gc.collect()

idx = np.arange(len(all_ys))

if groups is not None:
    # 80 / 10 / 10, no protein appearing in more than one split
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_idx, rem_idx = next(gss.split(idx, all_ys.numpy(), groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    rel_val, rel_test = next(gss2.split(rem_idx, all_ys.numpy()[rem_idx],
                                        groups[rem_idx]))
    val_idx, test_idx = rem_idx[rel_val], rem_idx[rel_test]

    assert not (set(groups[train_idx]) & set(groups[test_idx])), \
        "protein leakage between train and test"
else:
    train_idx, rem_idx = train_test_split(
        idx, test_size=0.20, random_state=SEED, stratify=all_ys.numpy())
    val_idx, test_idx = train_test_split(
        rem_idx, test_size=0.50, random_state=SEED,
        stratify=all_ys.numpy()[rem_idx])

train_embs, train_ys = all_embs[train_idx], all_ys[train_idx]
val_embs,   val_ys   = all_embs[val_idx],   all_ys[val_idx]
test_embs,  test_ys  = all_embs[test_idx],  all_ys[test_idx]

for name, y in (("train", train_ys), ("val", val_ys), ("test", test_ys)):
    n_pos = int((y == 1).sum())
    print(f"{name:5s}  n={len(y):6d}  pos={n_pos:6d}  neg={len(y) - n_pos:6d}")


class PTMDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i]


train_loader = DataLoader(PTMDataset(train_embs, train_ys),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(PTMDataset(val_embs, val_ys),
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(PTMDataset(test_embs, test_ys),
                          batch_size=BATCH_SIZE, shuffle=False)

# ---------------------------------------------------------------- model


class LearnablePositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_len, d_model))

    def forward(self, x):
        return x + self.pos_embedding


class ProteinTransformer(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, input_dim=INPUT_DIM,
                 hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.proj = nn.Linear(input_dim, hidden_dim)
        self.input_norm = nn.LayerNorm(hidden_dim)
        self.pos_encoding = LearnablePositionalEncoding(seq_len, hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=NHEAD,
            dim_feedforward=FF_DIM,
            dropout=DROPOUT,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer,
                                                 num_layers=NUM_LAYERS)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        # x: [B, 25, 1024]
        x = self.proj(x)
        x = self.input_norm(x)
        x = self.pos_encoding(x)
        x = self.transformer(x)             # [B, 25, hidden]

        center_index = x.shape[1] // 2
        center_feat = x[:, center_index, :]      # target residue
        mean_feat = x.mean(dim=1)                # window context

        x = torch.cat([center_feat, mean_feat], dim=1)
        return self.classifier(x).squeeze(-1)


model = ProteinTransformer().to(device)
print(f"trainable parameters: "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

pos_count = float((train_ys == 1).sum())
neg_count = float(len(train_ys) - pos_count)
imbalance = neg_count / max(pos_count, 1.0)
print(f"train imbalance: {imbalance:.2f}x negatives")

if BALANCE_CLASSES:
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([imbalance], device=device))
else:
    criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                              weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)

# ---------------------------------------------------------------- train / eval


def train_one_epoch():
    model.train()
    total = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.float().to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(train_loader)


@torch.no_grad()
def evaluate_probs(loader):
    model.eval()
    labels, probs = [], []
    for X, y in loader:
        p = torch.sigmoid(model(X.to(device)))
        labels.append(y.numpy())
        probs.append(p.cpu().numpy())
    return np.concatenate(labels), np.concatenate(probs)


def metrics(labels, probs, thresh):
    preds = (probs > thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        "ACC": (tp + tn) / max(tp + tn + fp + fn, 1),
        "SN":  tp / (tp + fn) if (tp + fn) else 0.0,
        "SP":  tn / (tn + fp) if (tn + fp) else 0.0,
        "MCC": matthews_corrcoef(labels, preds),
        "AUC": roc_auc_score(labels, probs),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
    }


# -2.0 not 0: an MCC of exactly 0 in epoch 1 would otherwise never checkpoint
best_mcc, counter = -2.0, 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch()

    val_labels, val_probs = evaluate_probs(val_loader)
    val_preds = (val_probs > 0.5).astype(int)
    val_mcc = matthews_corrcoef(val_labels, val_preds)
    val_auc = roc_auc_score(val_labels, val_probs)

    print(f"Epoch {epoch+1:3d}  |  Train Loss: {train_loss:.4f}  |  "
          f"Val MCC: {val_mcc:.4f}  |  Val AUC: {val_auc:.4f}")

    scheduler.step()

    if val_mcc > best_mcc:
        best_mcc, counter = val_mcc, 0
        torch.save(model.state_dict(), CKPT)
        print(f"  >>> best model saved (MCC {best_mcc:.4f})")
    else:
        counter += 1
        if counter >= PATIENCE:
            print("early stopping triggered")
            break

model.load_state_dict(torch.load(CKPT))
print(f"\nloaded best model — val MCC {best_mcc:.4f}")

# ---------------------------------------------------------------- threshold

val_labels, val_probs = evaluate_probs(val_loader)

best_thresh, best_val_mcc = 0.5, -2.0
for t in np.linspace(0.1, 0.9, 200):
    mcc = matthews_corrcoef(val_labels, (val_probs > t).astype(int))
    if mcc > best_val_mcc:
        best_val_mcc, best_thresh = mcc, t

print(f"best threshold on val: {best_thresh:.4f}  (val MCC {best_val_mcc:.4f})")

# ---------------------------------------------------------------- test

test_labels, test_probs = evaluate_probs(test_loader)

for name, thr in (("threshold 0.5", 0.5),
                  ("val-tuned threshold", best_thresh)):
    m = metrics(test_labels, test_probs, thr)
    print(f"\n===== FINAL TEST RESULTS — {name} ({thr:.4f}) =====")
    print(f"ACC: {m['ACC']:.4f}")
    print(f"SN:  {m['SN']:.4f}")
    print(f"SP:  {m['SP']:.4f}")
    print(f"MCC: {m['MCC']:.4f}")
    print(f"AUC: {m['AUC']:.4f}")
    print(f"TP: {m['TP']}  FP: {m['FP']}")
    print(f"FN: {m['FN']}  TN: {m['TN']}")

GPU: NVIDIA A100-SXM4-80GB  (85.09 GB)
Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
windows with 25 residues: 45902 / 45933
dataset shape: (45902, 25, 1024)
grouping by 'unique_id': 45902 proteins
train  n= 36721  pos= 18496  neg= 18225
val    n=  4590  pos=  2275  neg=  2315
test   n=  4591  pos=  2292  neg=  2299
trainable parameters: 7,109,633
train imbalance: 0.99x negatives
Epoch   1  |  Train Loss: 0.5745  |  Val MCC: 0.4675  |  Val AUC: 0.8072
  >>> best model saved (MCC 0.4675)
Epoch   2  |  Train Loss: 0.5091  |  Val MCC: 0.4919  |  Val AUC: 0.8221
  >>> best model saved (MCC 0.4919)
Epoch   3  |  Train Loss: 0.4597  |  Val MCC: 0.4953  |  Val AUC: 0.8305
  >>> best model saved (MCC 0.4953)
Epoch   4  |  Train Loss: 0.3984  |  Val MCC: 0.5020  |  Val AUC: 0.8290
  >>> best model saved (MCC 0.5020)
Epoch   5  |  Train Loss: 0.3343  |  Val MCC: 0.5104  |  Val AUC: 0.8300
  >>> 

In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.8 MB/s eta 0:00:00


In [ ]:
"""
Acetylation PTM prediction from ProtT5 embeddings — masked ProteinTransformer.

Model:
    Linear(1024 -> 256) -> LayerNorm
    -> token dropout (p=0.15, target site protected)
    -> + learned positions (init std 0.02)
    -> TransformerEncoder(2 layers, 4 heads, ff=512, pre-LN, gelu)
       with src_key_padding_mask over zero-padded residues
    -> LayerNorm
    -> concat[encoded center, masked attention pool, raw pre-encoder center]
       (3 x 256 = 768)
    -> LayerNorm -> Dropout -> Linear(768 -> 128) -> GELU -> Dropout
       -> Linear(128 -> 1)

Switches at the top:
    BALANCE_CLASSES  pos_weight in the loss (True keeps the old behaviour)
    GROUPED_SPLIT    split by protein instead of at random
"""

import gc
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import matthews_corrcoef, roc_auc_score, confusion_matrix

# ---------------------------------------------------------------- config

FILE_1 = "/content/drive/MyDrive/Acetylation/subash_with_all_csv_file_present_final_acetylation_graph.pt"
FILE_2 = "/content/drive/MyDrive/Acetylation/subash_with_all_csv_file_present_final_acetylation_graph_after_26169_line.pt"
CKPT = "best_model_masked_transformer.pt"

SEQ_LEN       = 25
INPUT_DIM     = 1024
HIDDEN        = 256
NLAYERS       = 2
NHEAD         = 4
DROPOUT       = 0.2
TOKEN_DROPOUT = 0.15

BATCH_SIZE   = 64
LR           = 1e-4
WEIGHT_DECAY = 1e-4
T_MAX        = 30
NUM_EPOCHS   = 30
PATIENCE     = 5

BALANCE_CLASSES = True    # False -> plain BCEWithLogitsLoss
GROUPED_SPLIT   = True    # False -> old random stratified split
GROUP_ATTR      = "unique_id"

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)")
print("Using device:", device)

from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------- data

data1 = torch.load(FILE_1, weights_only=False)
data2 = torch.load(FILE_2, weights_only=False)
all_data = data1[0] + data2[0]

valid_data = [item for item in all_data if item.emb.shape[0] == SEQ_LEN]
print(f"windows with {SEQ_LEN} residues: {len(valid_data)} / {len(all_data)}")

all_embs = torch.stack([item.emb for item in valid_data]).float()
all_ys = torch.tensor([item.y for item in valid_data]).float().view(-1)
print("dataset shape:", tuple(all_embs.shape))

# how much padding is actually present — the mask is a no-op if this is 0
pad_rows = (all_embs.abs().sum(-1) == 0)
print(f"zero-padded residue rows: {int(pad_rows.sum())} "
      f"({100 * pad_rows.float().mean():.2f}% of positions), "
      f"windows containing padding: {int(pad_rows.any(dim=1).sum())}")


def protein_of(item):
    """Strip the site suffix off an id like 'P12345_128' -> 'P12345'."""
    raw = str(getattr(item, GROUP_ATTR))
    return raw.rsplit("_", 1)[0] if "_" in raw else raw


groups = None
if GROUPED_SPLIT:
    if hasattr(valid_data[0], GROUP_ATTR):
        groups = np.array([protein_of(it) for it in valid_data])
        print(f"grouping by '{GROUP_ATTR}': {len(np.unique(groups))} proteins")
    else:
        print(f"WARNING: items have no '{GROUP_ATTR}' — falling back to "
              f"random split. Set GROUP_ATTR to the right field name.")

del all_data, data1, data2
gc.collect()

idx = np.arange(len(all_ys))

if groups is not None:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_idx, rem_idx = next(gss.split(idx, all_ys.numpy(), groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    rel_val, rel_test = next(gss2.split(rem_idx, all_ys.numpy()[rem_idx],
                                        groups[rem_idx]))
    val_idx, test_idx = rem_idx[rel_val], rem_idx[rel_test]

    assert not (set(groups[train_idx]) & set(groups[test_idx])), \
        "protein leakage between train and test"
else:
    train_idx, rem_idx = train_test_split(
        idx, test_size=0.20, random_state=SEED, stratify=all_ys.numpy())
    val_idx, test_idx = train_test_split(
        rem_idx, test_size=0.50, random_state=SEED,
        stratify=all_ys.numpy()[rem_idx])

train_embs, train_ys = all_embs[train_idx], all_ys[train_idx]
val_embs,   val_ys   = all_embs[val_idx],   all_ys[val_idx]
test_embs,  test_ys  = all_embs[test_idx],  all_ys[test_idx]

for name, y in (("train", train_ys), ("val", val_ys), ("test", test_ys)):
    n_pos = int((y == 1).sum())
    print(f"{name:5s}  n={len(y):6d}  pos={n_pos:6d}  neg={len(y) - n_pos:6d}")


class PTMDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i]


train_loader = DataLoader(PTMDataset(train_embs, train_ys),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(PTMDataset(val_embs, val_ys),
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(PTMDataset(test_embs, test_ys),
                          batch_size=BATCH_SIZE, shuffle=False)

# ---------------------------------------------------------------- model


class ProteinTransformer(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, input_dim=INPUT_DIM, hidden=HIDDEN,
                 nlayers=NLAYERS, nhead=NHEAD, dropout=DROPOUT,
                 token_dropout=TOKEN_DROPOUT):
        super().__init__()
        self.center = seq_len // 2
        self.token_dropout = token_dropout

        self.proj = nn.Linear(input_dim, hidden)
        self.in_norm = nn.LayerNorm(hidden)
        # small init: at std 1.0 the positional signal swamps the normalised input
        self.pos = nn.Parameter(torch.randn(1, seq_len, hidden) * 0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=nhead, dim_feedforward=hidden * 2,
            dropout=dropout, batch_first=True, activation="gelu",
            norm_first=True)                      # pre-LN: stable without warmup tricks
        self.enc = nn.TransformerEncoder(layer, nlayers)
        self.out_norm = nn.LayerNorm(hidden)
        self.attn = nn.Linear(hidden, 1)

        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 3),
            nn.Dropout(dropout),
            nn.Linear(hidden * 3, 128), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1))

    def forward(self, x):
        c = self.center

        # windows at sequence termini are zero-padded; without a mask the
        # encoder attends to padding as if it were residue content
        pad = (x.abs().sum(-1) == 0)
        pad[:, c] = False

        h = self.in_norm(self.proj(x))

        if self.training and self.token_dropout > 0:
            keep = torch.rand(h.shape[:2], device=h.device) > self.token_dropout
            keep[:, c] = True                     # never drop the target site
            h = h * keep.unsqueeze(-1)

        raw_center = h[:, c, :]                   # pre-attention shortcut
        h = self.enc(h + self.pos, src_key_padding_mask=pad)
        h = self.out_norm(h)

        w = self.attn(h).masked_fill(pad.unsqueeze(-1), float("-inf"))
        pooled = (h * torch.softmax(w, dim=1)).sum(dim=1)

        z = torch.cat([h[:, c, :], pooled, raw_center], dim=1)
        return self.head(z).squeeze(-1)


model = ProteinTransformer().to(device)
print(f"trainable parameters: "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

pos_count = float((train_ys == 1).sum())
neg_count = float(len(train_ys) - pos_count)
imbalance = neg_count / max(pos_count, 1.0)
print(f"train imbalance: {imbalance:.2f}x negatives")

if BALANCE_CLASSES:
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([imbalance], device=device))
else:
    criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                              weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)

# ---------------------------------------------------------------- train / eval


def train_one_epoch():
    model.train()
    total = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.float().to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(train_loader)


@torch.no_grad()
def evaluate_probs(loader):
    model.eval()
    labels, probs = [], []
    for X, y in loader:
        p = torch.sigmoid(model(X.to(device)))
        labels.append(y.numpy())
        probs.append(p.cpu().numpy())
    return np.concatenate(labels), np.concatenate(probs)


def metrics(labels, probs, thresh):
    preds = (probs > thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        "ACC": (tp + tn) / max(tp + tn + fp + fn, 1),
        "SN":  tp / (tp + fn) if (tp + fn) else 0.0,
        "SP":  tn / (tn + fp) if (tn + fp) else 0.0,
        "MCC": matthews_corrcoef(labels, preds),
        "AUC": roc_auc_score(labels, probs),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
    }


# -2.0 not 0: an MCC of exactly 0 in epoch 1 would otherwise never checkpoint
best_mcc, counter = -2.0, 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch()

    val_labels, val_probs = evaluate_probs(val_loader)
    val_preds = (val_probs > 0.5).astype(int)
    val_mcc = matthews_corrcoef(val_labels, val_preds)
    val_auc = roc_auc_score(val_labels, val_probs)

    print(f"Epoch {epoch+1:3d}  |  Train Loss: {train_loss:.4f}  |  "
          f"Val MCC: {val_mcc:.4f}  |  Val AUC: {val_auc:.4f}")

    scheduler.step()

    if val_mcc > best_mcc:
        best_mcc, counter = val_mcc, 0
        torch.save(model.state_dict(), CKPT)
        print(f"  >>> best model saved (MCC {best_mcc:.4f})")
    else:
        counter += 1
        if counter >= PATIENCE:
            print("early stopping triggered")
            break

model.load_state_dict(torch.load(CKPT))
print(f"\nloaded best model — val MCC {best_mcc:.4f}")

# ---------------------------------------------------------------- threshold

val_labels, val_probs = evaluate_probs(val_loader)

best_thresh, best_val_mcc = 0.5, -2.0
for t in np.linspace(0.1, 0.9, 200):
    mcc = matthews_corrcoef(val_labels, (val_probs > t).astype(int))
    if mcc > best_val_mcc:
        best_val_mcc, best_thresh = mcc, t

print(f"best threshold on val: {best_thresh:.4f}  (val MCC {best_val_mcc:.4f})")

# ---------------------------------------------------------------- test

test_labels, test_probs = evaluate_probs(test_loader)

for name, thr in (("threshold 0.5", 0.5),
                  ("val-tuned threshold", best_thresh)):
    m = metrics(test_labels, test_probs, thr)
    print(f"\n===== FINAL TEST RESULTS — {name} ({thr:.4f}) =====")
    print(f"ACC: {m['ACC']:.4f}")
    print(f"SN:  {m['SN']:.4f}")
    print(f"SP:  {m['SP']:.4f}")
    print(f"MCC: {m['MCC']:.4f}")
    print(f"AUC: {m['AUC']:.4f}")
    print(f"TP: {m['TP']}  FP: {m['FP']}")
    print(f"FN: {m['FN']}  TN: {m['TN']}")

GPU: NVIDIA A100-SXM4-80GB  (85.09 GB)
Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
windows with 25 residues: 45902 / 45933
dataset shape: (45902, 25, 1024)
zero-padded residue rows: 0 (0.00% of positions), windows containing padding: 0
grouping by 'unique_id': 45902 proteins
train  n= 36721  pos= 18496  neg= 18225
val    n=  4590  pos=  2275  neg=  2315
test   n=  4591  pos=  2292  neg=  2299


/tmp/ipykernel_20779/719006491.py:175: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, nlayers)


trainable parameters: 1,424,386
train imbalance: 0.99x negatives
Epoch   1  |  Train Loss: 0.5792  |  Val MCC: 0.4218  |  Val AUC: 0.7875
  >>> best model saved (MCC 0.4218)
Epoch   2  |  Train Loss: 0.5302  |  Val MCC: 0.4477  |  Val AUC: 0.8050
  >>> best model saved (MCC 0.4477)
Epoch   3  |  Train Loss: 0.4998  |  Val MCC: 0.4658  |  Val AUC: 0.8092
  >>> best model saved (MCC 0.4658)
Epoch   4  |  Train Loss: 0.4720  |  Val MCC: 0.4887  |  Val AUC: 0.8181
  >>> best model saved (MCC 0.4887)
Epoch   5  |  Train Loss: 0.4352  |  Val MCC: 0.4889  |  Val AUC: 0.8206
  >>> best model saved (MCC 0.4889)
Epoch   6  |  Train Loss: 0.3924  |  Val MCC: 0.4794  |  Val AUC: 0.8128
Epoch   7  |  Train Loss: 0.3461  |  Val MCC: 0.4641  |  Val AUC: 0.8109
Epoch   8  |  Train Loss: 0.2957  |  Val MCC: 0.4667  |  Val AUC: 0.8060
Epoch   9  |  Train Loss: 0.2527  |  Val MCC: 0.4607  |  Val AUC: 0.7996
Epoch  10  |  Train Loss: 0.2212  |  Val MCC: 0.4344  |  Val AUC: 0.7945
early stopping triggered


In [ ]:
"""
Acetylation PTM prediction from ProtT5 embeddings — masked ProteinTransformer.

Model:
    Linear(1024 -> 256) -> LayerNorm
    -> token dropout (p=0.15, target site protected)
    -> + learned positions (init std 0.02)
    -> TransformerEncoder(2 layers, 4 heads, ff=512, pre-LN, gelu)
       with src_key_padding_mask over zero-padded residues
    -> LayerNorm
    -> concat[encoded center, masked attention pool, raw pre-encoder center]
       (3 x 256 = 768)
    -> LayerNorm -> Dropout -> Linear(768 -> 128) -> GELU -> Dropout
       -> Linear(128 -> 1)

Switches at the top:
    BALANCE_CLASSES  pos_weight in the loss (True keeps the old behaviour)
    GROUPED_SPLIT    split by protein instead of at random
"""

import gc
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import matthews_corrcoef, roc_auc_score, confusion_matrix

# ---------------------------------------------------------------- config

FILE_1 = "/content/drive/MyDrive/Acetylation/subash_with_all_csv_file_present_final_acetylation_graph.pt"
FILE_2 = "/content/drive/MyDrive/Acetylation/subash_with_all_csv_file_present_final_acetylation_graph_after_26169_line.pt"
CKPT = "best_model_masked_transformer.pt"

SEQ_LEN       = 25
INPUT_DIM     = 1024
HIDDEN        = 256
NLAYERS       = 2
NHEAD         = 4
DROPOUT       = 0.2
TOKEN_DROPOUT = 0.15

BATCH_SIZE   = 64
LR           = 1e-4
WEIGHT_DECAY = 1e-4
T_MAX        = 30
NUM_EPOCHS   = 30
PATIENCE     = 5

BALANCE_CLASSES = True    # False -> plain BCEWithLogitsLoss
GROUPED_SPLIT   = True    # False -> old random stratified split
GROUP_ATTR      = "unique_id"

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)")
print("Using device:", device)

from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------- data

data1 = torch.load(FILE_1, weights_only=False)
data2 = torch.load(FILE_2, weights_only=False)
all_data = data1[0] + data2[0]

valid_data = [item for item in all_data if item.emb.shape[0] == SEQ_LEN]
print(f"windows with {SEQ_LEN} residues: {len(valid_data)} / {len(all_data)}")

all_embs = torch.stack([item.emb for item in valid_data]).float()
all_ys = torch.tensor([item.y for item in valid_data]).float().view(-1)
print("dataset shape:", tuple(all_embs.shape))

# how much padding is actually present — the mask is a no-op if this is 0
pad_rows = (all_embs.abs().sum(-1) == 0)
print(f"zero-padded residue rows: {int(pad_rows.sum())} "
      f"({100 * pad_rows.float().mean():.2f}% of positions), "
      f"windows containing padding: {int(pad_rows.any(dim=1).sum())}")


def protein_of(item):
    """Strip the site suffix off an id like 'P12345_128' -> 'P12345'."""
    raw = str(getattr(item, GROUP_ATTR))
    return raw.rsplit("_", 1)[0] if "_" in raw else raw


groups = None
if GROUPED_SPLIT:
    if hasattr(valid_data[0], GROUP_ATTR):
        groups = np.array([protein_of(it) for it in valid_data])
        print(f"grouping by '{GROUP_ATTR}': {len(np.unique(groups))} proteins")
    else:
        print(f"WARNING: items have no '{GROUP_ATTR}' — falling back to "
              f"random split. Set GROUP_ATTR to the right field name.")

del all_data, data1, data2
gc.collect()

idx = np.arange(len(all_ys))

if groups is not None:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_idx, rem_idx = next(gss.split(idx, all_ys.numpy(), groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    rel_val, rel_test = next(gss2.split(rem_idx, all_ys.numpy()[rem_idx],
                                        groups[rem_idx]))
    val_idx, test_idx = rem_idx[rel_val], rem_idx[rel_test]

    assert not (set(groups[train_idx]) & set(groups[test_idx])), \
        "protein leakage between train and test"
else:
    train_idx, rem_idx = train_test_split(
        idx, test_size=0.20, random_state=SEED, stratify=all_ys.numpy())
    val_idx, test_idx = train_test_split(
        rem_idx, test_size=0.50, random_state=SEED,
        stratify=all_ys.numpy()[rem_idx])

train_embs, train_ys = all_embs[train_idx], all_ys[train_idx]
val_embs,   val_ys   = all_embs[val_idx],   all_ys[val_idx]
test_embs,  test_ys  = all_embs[test_idx],  all_ys[test_idx]

for name, y in (("train", train_ys), ("val", val_ys), ("test", test_ys)):
    n_pos = int((y == 1).sum())
    print(f"{name:5s}  n={len(y):6d}  pos={n_pos:6d}  neg={len(y) - n_pos:6d}")


class PTMDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i]


train_loader = DataLoader(PTMDataset(train_embs, train_ys),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(PTMDataset(val_embs, val_ys),
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(PTMDataset(test_embs, test_ys),
                          batch_size=BATCH_SIZE, shuffle=False)

# ---------------------------------------------------------------- model


class ProteinTransformer(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, input_dim=INPUT_DIM, hidden=HIDDEN,
                 nlayers=NLAYERS, nhead=NHEAD, dropout=DROPOUT,
                 token_dropout=TOKEN_DROPOUT):
        super().__init__()
        self.center = seq_len // 2
        self.token_dropout = token_dropout

        self.proj = nn.Linear(input_dim, hidden)
        self.in_norm = nn.LayerNorm(hidden)
        # small init: at std 1.0 the positional signal swamps the normalised input
        self.pos = nn.Parameter(torch.randn(1, seq_len, hidden) * 0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=nhead, dim_feedforward=hidden * 2,
            dropout=dropout, batch_first=True, activation="gelu",
            norm_first=True)                      # pre-LN: stable without warmup tricks
        self.enc = nn.TransformerEncoder(layer, nlayers)
        self.out_norm = nn.LayerNorm(hidden)
        self.attn = nn.Linear(hidden, 1)

        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 3),
            nn.Dropout(dropout),
            nn.Linear(hidden * 3, 128), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1))

    def forward(self, x):
        c = self.center

        # windows at sequence termini are zero-padded; without a mask the
        # encoder attends to padding as if it were residue content
        pad = (x.abs().sum(-1) == 0)
        pad[:, c] = False

        h = self.in_norm(self.proj(x))

        if self.training and self.token_dropout > 0:
            keep = torch.rand(h.shape[:2], device=h.device) > self.token_dropout
            keep[:, c] = True                     # never drop the target site
            h = h * keep.unsqueeze(-1)

        raw_center = h[:, c, :]                   # pre-attention shortcut
        h = self.enc(h + self.pos, src_key_padding_mask=pad)
        h = self.out_norm(h)

        w = self.attn(h).masked_fill(pad.unsqueeze(-1), float("-inf"))
        pooled = (h * torch.softmax(w, dim=1)).sum(dim=1)

        z = torch.cat([h[:, c, :], pooled, raw_center], dim=1)
        return self.head(z).squeeze(-1)


model = ProteinTransformer().to(device)
print(f"trainable parameters: "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

pos_count = float((train_ys == 1).sum())
neg_count = float(len(train_ys) - pos_count)
imbalance = neg_count / max(pos_count, 1.0)
print(f"train imbalance: {imbalance:.2f}x negatives")

if BALANCE_CLASSES:
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([imbalance], device=device))
else:
    criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                              weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)

# ---------------------------------------------------------------- train / eval


def train_one_epoch():
    model.train()
    total = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.float().to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(train_loader)


@torch.no_grad()
def evaluate_probs(loader):
    model.eval()
    labels, probs = [], []
    for X, y in loader:
        p = torch.sigmoid(model(X.to(device)))
        labels.append(y.numpy())
        probs.append(p.cpu().numpy())
    return np.concatenate(labels), np.concatenate(probs)


def metrics(labels, probs, thresh):
    preds = (probs > thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        "ACC": (tp + tn) / max(tp + tn + fp + fn, 1),
        "SN":  tp / (tp + fn) if (tp + fn) else 0.0,
        "SP":  tn / (tn + fp) if (tn + fp) else 0.0,
        "MCC": matthews_corrcoef(labels, preds),
        "AUC": roc_auc_score(labels, probs),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
    }


# -2.0 not 0: an MCC of exactly 0 in epoch 1 would otherwise never checkpoint
best_mcc, counter = -2.0, 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch()

    val_labels, val_probs = evaluate_probs(val_loader)
    val_preds = (val_probs > 0.5).astype(int)
    val_mcc = matthews_corrcoef(val_labels, val_preds)
    val_auc = roc_auc_score(val_labels, val_probs)

    print(f"Epoch {epoch+1:3d}  |  Train Loss: {train_loss:.4f}  |  "
          f"Val MCC: {val_mcc:.4f}  |  Val AUC: {val_auc:.4f}")

    scheduler.step()

    if val_mcc > best_mcc:
        best_mcc, counter = val_mcc, 0
        torch.save(model.state_dict(), CKPT)
        print(f"  >>> best model saved (MCC {best_mcc:.4f})")
    else:
        counter += 1
        if counter >= PATIENCE:
            print("early stopping triggered")
            break

model.load_state_dict(torch.load(CKPT))
print(f"\nloaded best model — val MCC {best_mcc:.4f}")

# ---------------------------------------------------------------- threshold

val_labels, val_probs = evaluate_probs(val_loader)

best_thresh, best_val_mcc = 0.5, -2.0
for t in np.linspace(0.1, 0.9, 200):
    mcc = matthews_corrcoef(val_labels, (val_probs > t).astype(int))
    if mcc > best_val_mcc:
        best_val_mcc, best_thresh = mcc, t

print(f"best threshold on val: {best_thresh:.4f}  (val MCC {best_val_mcc:.4f})")

# ---------------------------------------------------------------- test

test_labels, test_probs = evaluate_probs(test_loader)

for name, thr in (("threshold 0.5", 0.5),
                  ("val-tuned threshold", best_thresh)):
    m = metrics(test_labels, test_probs, thr)
    print(f"\n===== FINAL TEST RESULTS — {name} ({thr:.4f}) =====")
    print(f"ACC: {m['ACC']:.4f}")
    print(f"SN:  {m['SN']:.4f}")
    print(f"SP:  {m['SP']:.4f}")
    print(f"MCC: {m['MCC']:.4f}")
    print(f"AUC: {m['AUC']:.4f}")
    print(f"TP: {m['TP']}  FP: {m['FP']}")
    print(f"FN: {m['FN']}  TN: {m['TN']}")

GPU: NVIDIA A100-SXM4-80GB  (85.09 GB)
Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
windows with 25 residues: 45902 / 45933
dataset shape: (45902, 25, 1024)
zero-padded residue rows: 0 (0.00% of positions), windows containing padding: 0
grouping by 'unique_id': 45902 proteins
train  n= 36721  pos= 18496  neg= 18225
val    n=  4590  pos=  2275  neg=  2315
test   n=  4591  pos=  2292  neg=  2299


/tmp/ipykernel_20779/719006491.py:175: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, nlayers)


trainable parameters: 1,424,386
train imbalance: 0.99x negatives
Epoch   1  |  Train Loss: 0.5792  |  Val MCC: 0.4218  |  Val AUC: 0.7875
  >>> best model saved (MCC 0.4218)
Epoch   2  |  Train Loss: 0.5302  |  Val MCC: 0.4477  |  Val AUC: 0.8050
  >>> best model saved (MCC 0.4477)
Epoch   3  |  Train Loss: 0.4998  |  Val MCC: 0.4658  |  Val AUC: 0.8092
  >>> best model saved (MCC 0.4658)
Epoch   4  |  Train Loss: 0.4720  |  Val MCC: 0.4887  |  Val AUC: 0.8181
  >>> best model saved (MCC 0.4887)
Epoch   5  |  Train Loss: 0.4352  |  Val MCC: 0.4889  |  Val AUC: 0.8206
  >>> best model saved (MCC 0.4889)
Epoch   6  |  Train Loss: 0.3924  |  Val MCC: 0.4794  |  Val AUC: 0.8128
Epoch   7  |  Train Loss: 0.3461  |  Val MCC: 0.4641  |  Val AUC: 0.8109
Epoch   8  |  Train Loss: 0.2957  |  Val MCC: 0.4667  |  Val AUC: 0.8060
Epoch   9  |  Train Loss: 0.2527  |  Val MCC: 0.4607  |  Val AUC: 0.7996
Epoch  10  |  Train Loss: 0.2212  |  Val MCC: 0.4344  |  Val AUC: 0.7945
early stopping triggered


In [ ]:
import gc
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# ---- config: must match your training script ----
FILE_1 = "/content/drive/MyDrive/Acetylation/subash_with_all_csv_file_present_final_acetylation_graph.pt"
FILE_2 = "/content/drive/MyDrive/Acetylation/subash_with_all_csv_file_present_final_acetylation_graph_after_26169_line.pt"

SEQ_LEN       = 25
SEED          = 7
GROUPED_SPLIT = True
GROUP_ATTR    = "unique_id"

from google.colab import drive
drive.mount("/content/drive")

# ---- load ----
data1 = torch.load(FILE_1, weights_only=False)
data2 = torch.load(FILE_2, weights_only=False)
all_data = data1[0] + data2[0]

valid_data = [item for item in all_data if item.emb.shape[0] == SEQ_LEN]
print(f"windows loaded:  {len(all_data):,}")
print(f"windows kept:    {len(valid_data):,}")
print(f"windows dropped: {len(all_data) - len(valid_data):,}")

all_ys = torch.tensor([item.y for item in valid_data]).float().view(-1).numpy()

# ---- protein groups ----
def protein_of(item):
    """Strip the site suffix off an id like 'P12345_128' -> 'P12345'."""
    raw = str(getattr(item, GROUP_ATTR))
    return raw.rsplit("_", 1)[0] if "_" in raw else raw

groups = None
if GROUPED_SPLIT:
    if hasattr(valid_data[0], GROUP_ATTR):
        groups = np.array([protein_of(it) for it in valid_data])
        print(f"\nexample id: {getattr(valid_data[0], GROUP_ATTR)}  ->  protein: {groups[0]}")
    else:
        print(f"WARNING: no '{GROUP_ATTR}' attribute — random split will be used")

del all_data, data1, data2
gc.collect()

# ---- whole dataset ----
n_pos_all = int((all_ys == 1).sum())
n_neg_all = len(all_ys) - n_pos_all
print("\nWHOLE DATASET")
print(f"  samples    {len(all_ys):,}")
print(f"  positives  {n_pos_all:,}  ({100 * n_pos_all / len(all_ys):.2f}%)")
print(f"  negatives  {n_neg_all:,}")
print(f"  ratio      {n_neg_all / max(n_pos_all, 1):.1f} negatives per positive")
if groups is not None:
    print(f"  proteins   {len(np.unique(groups)):,}")
    print(f"  proteins with >=1 positive: {len(np.unique(groups[all_ys == 1])):,}")

# ---- the split (identical to the training script) ----
idx = np.arange(len(all_ys))

if groups is not None:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_idx, rem_idx = next(gss.split(idx, all_ys, groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    rel_val, rel_test = next(gss2.split(rem_idx, all_ys[rem_idx], groups[rem_idx]))
    val_idx, test_idx = rem_idx[rel_val], rem_idx[rel_test]
else:
    train_idx, rem_idx = train_test_split(
        idx, test_size=0.20, random_state=SEED, stratify=all_ys)
    val_idx, test_idx = train_test_split(
        rem_idx, test_size=0.50, random_state=SEED, stratify=all_ys[rem_idx])

splits = {"train": train_idx, "val": val_idx, "test": test_idx}

# ---- the table ----
rows = []
for name, sel in splits.items():
    y = all_ys[sel]
    n = len(y)
    n_pos = int((y == 1).sum())
    n_neg = n - n_pos
    row = {
        "split": name,
        "samples": n,
        "pct_of_total": round(100 * n / len(all_ys), 1),
        "positives": n_pos,
        "negatives": n_neg,
        "pos_rate_pct": round(100 * n_pos / max(n, 1), 2),
        "neg_per_pos": round(n_neg / max(n_pos, 1), 1),
    }
    if groups is not None:
        g = groups[sel]
        row["proteins"] = len(np.unique(g))
        row["proteins_with_pos"] = len(np.unique(g[y == 1]))
    rows.append(row)

stats = pd.DataFrame(rows).set_index("split")
print("\n" + stats.to_string())

# ---- leakage check ----
if groups is not None:
    sets = {k: set(groups[v].tolist()) for k, v in splits.items()}
    print("\nprotein overlap between splits (all must be 0)")
    clean = True
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        shared = len(sets[a] & sets[b])
        if shared:
            clean = False
        print(f"  {a:>5} n {b:<5} {shared:>6}   {'OK' if shared == 0 else 'LEAK'}")
    print("\nno protein appears in more than one split" if clean
          else "\nWARNING: protein leakage detected")

stats.to_csv("split_stats.csv")

Mounted at /content/drive
windows loaded:  45,933
windows kept:    45,902
windows dropped: 31

example id: O00116;102;658;1  ->  protein: O00116;102;658;1

WHOLE DATASET
  samples    45,902
  positives  23,063  (50.24%)
  negatives  22,839
  ratio      1.0 negatives per positive
  proteins   45,902
  proteins with >=1 positive: 23,063

       samples  pct_of_total  positives  negatives  pos_rate_pct  neg_per_pos  proteins  proteins_with_pos
split                                                                                                     
train    36721          80.0      18496      18225         50.37          1.0     36721              18496
val       4590          10.0       2275       2315         49.56          1.0      4590               2275
test      4591          10.0       2292       2299         49.92          1.0      4591               2292

protein overlap between splits (all must be 0)
  train n val        0   OK
  train n test       0   OK
    val n test       0 